# pinball_endpoint

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Each setting's agents: the large buffer against three agents at the setting's
# memory budget.
SETTINGS = {
    "medium": {
        "Large": "ddqn_pinball",
        "Small": "ddqn_medium_pinball",
        "Small 10-step": "ddqn_medium_nstep_pinball",
        "Endpoint": "endpoint_pinball",
    },
    "small": {
        "Large": "ddqn_pinball",
        "Small": "ddqn_small_pinball",
        "Small 10-step": "ddqn_small_nstep_pinball",
        "Endpoint": "endpoint_small_pinball",
    },
}

In [ ]:
# Load each component's stored runs once and compute its mean return over seeds
# with a 95% bootstrap CI.
names = {name for agents in SETTINGS.values() for name in agents.values()}
curves = {}
for name in names:
    runs = load_results(EXPERIMENT, name)
    if runs is not None:
        timesteps, returns = subsample(return_curve(runs.reward, runs.done))
        curves[name] = (timesteps, mean_ci(returns), len(returns))

In [ ]:
# Style for the learning-curve plots. Every piece of text shares one size.
FONT_SIZE = 32
STYLE = {
    "figure.figsize": (8, 8),
    "font.family": "Times New Roman",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
}
COLORS = {
    "ddqn_pinball": "black",
    "ddqn_medium_pinball": "tab:blue",
    "ddqn_medium_nstep_pinball": "tab:purple",
    "endpoint_pinball": "tab:red",
    "ddqn_small_pinball": "tab:blue",
    "ddqn_small_nstep_pinball": "tab:purple",
    "endpoint_small_pinball": "tab:red",
}
BAND_ALPHA = 0.2
# Both settings share colors, so only the medium plot carries the legend.
LEGEND = {
    "loc": "lower right",
    "bbox_to_anchor": (1.04, -0.04),
    "handlelength": 1,
    "handletextpad": 0.4,
}
LEGEND_MARKER = {"marker": "o", "linestyle": "none", "markersize": 14}
LEGEND_SETTINGS = {"medium"}
LEGEND_ORDER = ["Endpoint", "Large", "Small", "Small 10-step"]
# Memory relative to the 10k large-recency buffer.
TITLES = {"medium": "Pinball 10x Smaller", "small": "Pinball 20x Smaller"}
BOX_ASPECT = 1
# Ticks sit only at the axis ends, so the labels can tuck in between them.
LABEL_PAD = -20
XLABEL = "Time Steps"
XTICKS = [0, 100_000]
XTICK_LABELS = ["0", "100k"]
YLABEL = "Return"
YTICKS = [-1000, 0]
YTICK_LABELS = ["-1k", "0"]

In [ ]:
# Plot each setting's mean returns and CI bands over time.
figures = {}
for setting, agents in SETTINGS.items():
    drawn = {agent: name for agent, name in agents.items() if name in curves}
    if not drawn:
        continue
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots()
        ax.set_box_aspect(BOX_ASPECT)
        for name in drawn.values():
            timesteps, (mean, low, high), _ = curves[name]
            color = COLORS[name]
            ax.fill_between(timesteps, low, high, color=color, alpha=BAND_ALPHA)
            ax.plot(timesteps, mean, color=color)
        ax.set_title(TITLES[setting])
        ax.set_xlabel(XLABEL, labelpad=LABEL_PAD)
        ax.set_xticks(XTICKS, XTICK_LABELS)
        ax.set_ylabel(YLABEL, labelpad=LABEL_PAD)
        ax.set_yticks(YTICKS, YTICK_LABELS)
        ax.set_ylim(YTICKS[0], YTICKS[-1])
        fig.tight_layout()
        if setting in LEGEND_SETTINGS:
            shown = [agent for agent in LEGEND_ORDER if agent in drawn]
            handles = [
                Line2D([], [], color=COLORS[drawn[agent]], **LEGEND_MARKER)
                for agent in shown
            ]
            ax.legend(handles, shown, **LEGEND)
    figures[f"curves_{setting}"] = fig
plt.show()

In [ ]:
# Save every plotted figure as a PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    for suffix in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{suffix}", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")